In [9]:
import torch
import subprocess
import os

# Check if accelerate is available
try:
    import accelerate
    accelerate_available = True
except ImportError:
    accelerate_available = False

In [12]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Current CUDA device:", torch.cuda.current_device())
    print("CUDA device name:", torch.cuda.get_device_name(0))
print("Python executable:", __import__("sys").executable)
print("Accelerate available:", accelerate_available)

PyTorch version: 2.9.1+cu128
CUDA available: False
CUDA device count: 4
Python executable: /opt/conda/envs/my_env/bin/python
Accelerate available: True


In [6]:
# Check GPU processes and memory usage
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=memory.used,memory.total', '--format=csv,noheader,nounits'], 
                          capture_output=True, text=True, check=True)
    print("GPU Memory Usage (Used/Total MB):")
    for i, line in enumerate(result.stdout.strip().split('\n')):
        used, total = line.split(', ')
        print(f"GPU {i}: {used}/{total} MB")
except Exception as e:
    print(f"Error checking GPU memory: {e}")

print("\n" + "="*50)

# Check for running processes on GPU
try:
    result = subprocess.run(['nvidia-smi', '--query-compute-apps=pid,process_name,gpu_uuid,used_memory', '--format=csv,noheader'], 
                          capture_output=True, text=True, check=True)
    if result.stdout.strip():
        print("Processes using GPU:")
        print(result.stdout)
    else:
        print("No processes currently using GPU")
except Exception as e:
    print(f"Error checking GPU processes: {e}")

GPU Memory Usage (Used/Total MB):
GPU 0: 15/49140 MB
GPU 1: 15/49140 MB
GPU 2: 15/49140 MB
GPU 3: 15/49140 MB

No processes currently using GPU


In [7]:
# Test CUDA initialization
print("Testing CUDA initialization...")
try:
    # Try to create a simple tensor on CUDA
    if torch.cuda.device_count() > 0:
        device = torch.device("cuda:0")
        x = torch.tensor([1.0], device=device)
        print("✅ CUDA tensor creation successful!")
        print(f"Tensor on device: {x.device}")
    else:
        print("❌ No CUDA devices found")
except Exception as e:
    print(f"❌ CUDA initialization failed: {e}")

print("\n" + "="*50)

# Check CUDA driver version
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], 
                          capture_output=True, text=True, check=True)
    driver_version = result.stdout.strip().split('\n')[0]
    print(f"NVIDIA Driver Version: {driver_version}")
    
    # Check CUDA runtime version
    print(f"PyTorch CUDA Version: {torch.version.cuda}")
    
except Exception as e:
    print(f"Error checking driver version: {e}")

Testing CUDA initialization...
❌ CUDA initialization failed: CUDA driver initialization failed, you might not have a CUDA gpu.

NVIDIA Driver Version: 575.57.08
PyTorch CUDA Version: 12.8


In [12]:
# Check CUDA runtime version and compatibility
try:
    result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True, check=True)
    print("NVCC Version:")
    print(result.stdout)
except Exception as e:
    print(f"NVCC not found or error: {e}")

print("="*50)

# Check detailed nvidia-smi output
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=True)
    print("nvidia-smi output:")
    print(result.stdout)
except Exception as e:
    print(f"nvidia-smi error: {e}")

print("="*50)

# Check if this is a known PyTorch issue
print("PyTorch CUDA compilation details:")
print(f"PyTorch was compiled with CUDA {torch.version.cuda}")
print(f"Current CUDA runtime available: {torch.cuda.is_available()}")

# Try to get more specific error
try:
    torch.cuda.init()
    print("CUDA initialization successful!")
except Exception as e:
    print(f"CUDA init error: {e}")

NVCC not found or error: [Errno 2] No such file or directory: 'nvcc'
nvidia-smi output:
Tue Jan 13 08:31:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               On  |   00000000:81:00.0 Off |                  Off |
| 30%   30C    P8             21W /  300W |      15MiB /  49140MiB |      0%      Default |
|                                         |         

# CUDA Initialization Problem - Solutions

**Problem**: PyTorch compiled with CUDA 12.8, but system has CUDA 12.9 driver

## Solution Options:

### 1. **Reinstall PyTorch with compatible CUDA version** (Recommended)
```bash
# Uninstall current PyTorch
pip uninstall torch torchvision torchaudio

# Install PyTorch with CUDA 12.1 (more compatible)
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
```

### 2. **Use conda to manage CUDA dependencies**
```bash
conda install pytorch torchvision torchaudio pytorch-cuda=12.1 -c pytorch -c nvidia
```

### 3. **Check for CUDA runtime libraries**
- The issue might be missing CUDA runtime libraries in your environment
- Your container/environment may need CUDA runtime installed

### 4. **Temporary fix: Use CPU for now**
```python
device = torch.device("cpu")
```

### 5. **Restart your container/kernel** after PyTorch reinstall
- CUDA context can get stuck
- Fresh start often resolves driver initialization issues

In [13]:
# Since you were working with these versions hours ago, let's check what changed
print("=== INVESTIGATING SYSTEM STATE CHANGES ===\n")

# 1. Check if CUDA context is stuck/corrupted
print("1. Testing CUDA context reset...")
try:
    # Try to reset CUDA context
    if hasattr(torch.cuda, 'empty_cache'):
        torch.cuda.empty_cache()
        print("   ✅ CUDA cache cleared")
    
    if hasattr(torch.cuda, 'synchronize'):
        torch.cuda.synchronize()
        print("   ✅ CUDA synchronized")
        
except Exception as e:
    print(f"   ❌ CUDA context operations failed: {e}")

print()

# 2. Check environment variables that might affect CUDA
print("2. Checking CUDA environment variables...")
cuda_env_vars = ['CUDA_HOME', 'CUDA_PATH', 'CUDA_VISIBLE_DEVICES', 'NVIDIA_VISIBLE_DEVICES']
for var in cuda_env_vars:
    value = os.environ.get(var, 'Not set')
    print(f"   {var}: {value}")

print()

# 3. Check if this is a container/process isolation issue
print("3. Checking process and container state...")
try:
    # Check if running in container
    with open('/proc/1/cgroup', 'r') as f:
        cgroup_content = f.read()
        if 'docker' in cgroup_content or 'containerd' in cgroup_content:
            print("   ✅ Running in container")
        else:
            print("   ℹ️  Not in container")
except:
    print("   ❓ Could not determine container status")

# Check memory pressure
try:
    with open('/proc/meminfo', 'r') as f:
        meminfo = f.read()
        for line in meminfo.split('\n'):
            if 'MemAvailable' in line:
                available = int(line.split()[1]) // 1024  # Convert to MB
                print(f"   Available RAM: {available} MB")
                if available < 1000:
                    print("   ⚠️  Low memory detected!")
                break
except:
    pass

=== INVESTIGATING SYSTEM STATE CHANGES ===

1. Testing CUDA context reset...
   ✅ CUDA cache cleared
   ❌ CUDA context operations failed: CUDA driver initialization failed, you might not have a CUDA gpu.

2. Checking CUDA environment variables...
   CUDA_HOME: Not set
   CUDA_PATH: Not set
   CUDA_VISIBLE_DEVICES: Not set
   NVIDIA_VISIBLE_DEVICES: all

3. Checking process and container state...
   ✅ Running in container
   Available RAM: 50112 MB


In [14]:
# 4. Check CUDA device files and permissions (common container issue)
print("4. Checking CUDA device files...")
cuda_devices = ['/dev/nvidia0', '/dev/nvidia1', '/dev/nvidia2', '/dev/nvidia3', '/dev/nvidiactl', '/dev/nvidia-uvm']
for device in cuda_devices:
    try:
        if os.path.exists(device):
            stat_info = os.stat(device)
            print(f"   ✅ {device} exists (permissions: {oct(stat_info.st_mode)[-3:]})")
        else:
            print(f"   ❌ {device} missing")
    except Exception as e:
        print(f"   ❓ {device}: {e}")

print()

# 5. Check if nvidia-ml-py can access GPUs (different than PyTorch)
print("5. Testing direct NVIDIA access...")
try:
    import pynvml
    pynvml.nvmlInit()
    gpu_count = pynvml.nvmlDeviceGetCount()
    print(f"   ✅ NVML can see {gpu_count} GPUs")
    
    for i in range(gpu_count):
        handle = pynvml.nvmlDeviceGetHandleByIndex(i)
        name = pynvml.nvmlDeviceGetName(handle)
        print(f"   ✅ GPU {i}: {name}")
        
except ImportError:
    print("   ℹ️  pynvml not available, installing...")
    try:
        subprocess.run(['pip', 'install', 'nvidia-ml-py3'], check=True, capture_output=True)
        print("   ✅ pynvml installed, restart kernel to test")
    except:
        print("   ❌ Could not install pynvml")
except Exception as e:
    print(f"   ❌ NVML error: {e}")

print()

# 6. Check for recent system changes
print("6. Checking for recent system changes...")
try:
    # Check container runtime
    result = subprocess.run(['cat', '/proc/version'], capture_output=True, text=True, check=True)
    print(f"   Kernel: {result.stdout.strip()}")
except:
    pass

# Check if container was recently restarted
try:
    result = subprocess.run(['uptime'], capture_output=True, text=True, check=True)
    print(f"   System uptime: {result.stdout.strip()}")
except:
    pass

4. Checking CUDA device files...
   ✅ /dev/nvidia0 exists (permissions: 666)
   ✅ /dev/nvidia1 exists (permissions: 666)
   ✅ /dev/nvidia2 exists (permissions: 666)
   ✅ /dev/nvidia3 exists (permissions: 666)
   ✅ /dev/nvidiactl exists (permissions: 666)
   ✅ /dev/nvidia-uvm exists (permissions: 666)

5. Testing direct NVIDIA access...
   ✅ NVML can see 4 GPUs
   ✅ GPU 0: NVIDIA RTX A6000
   ✅ GPU 1: NVIDIA RTX A6000
   ✅ GPU 2: NVIDIA RTX A6000
   ✅ GPU 3: NVIDIA RTX A6000

6. Checking for recent system changes...
   Kernel: Linux version 5.4.0-216-generic (buildd@lcy02-amd64-014) (gcc version 9.4.0 (Ubuntu 9.4.0-1ubuntu1~20.04.2)) #236-Ubuntu SMP Fri Apr 11 19:53:21 UTC 2025
   System uptime: 08:34:08 up 55 days, 20:35,  0 users,  load average: 0.42, 0.65, 0.93


In [16]:
# 7. Check PyTorch's CUDA runtime dependencies
print("7. Diagnosing PyTorch CUDA runtime libraries...")

# Check what CUDA libraries PyTorch is looking for
import ctypes
import glob

try:
    # Try to load the CUDA runtime library that PyTorch uses
    print("   Checking CUDA runtime libraries...")
    
    # Common CUDA runtime library paths
    cuda_lib_paths = [
        '/usr/local/cuda*/lib64/libcudart.so*',
        '/opt/conda/envs/*/lib/libcudart.so*',
        '/usr/lib/x86_64-linux-gnu/libcudart.so*'
    ]
    
    found_libs = []
    for pattern in cuda_lib_paths:
        libs = glob.glob(pattern)
        found_libs.extend(libs)
    
    if found_libs:
        print("   ✅ Found CUDA runtime libraries:")
        for lib in found_libs[:5]:  # Show first 5
            print(f"      {lib}")
    else:
        print("   ❌ No CUDA runtime libraries found")
    
except Exception as e:
    print(f"   ❓ Library check error: {e}")

print()

# 8. Try to manually load PyTorch's CUDA backend
print("8. Testing PyTorch CUDA backend loading...")
try:
    # Try to access PyTorch's internal CUDA functions
    import torch._C._cuda
    print("   ✅ PyTorch CUDA C extension loaded")
    
    # Try CUDA device count from C extension
    device_count = torch._C._cuda.device_count()
    print(f"   ✅ C extension sees {device_count} devices")
    
except Exception as e:
    print(f"   ❌ PyTorch CUDA backend error: {e}")
    print(f"   ❌ This suggests PyTorch CUDA bindings are broken")

print()

# 9. Check if restarting the Python process would help
print("9. Potential solutions based on diagnosis:")
print("   💡 Since NVIDIA drivers work but PyTorch CUDA fails:")
print("   💡 This is likely a PyTorch CUDA runtime library issue")
print("   💡 Solutions to try:")
print("   1️⃣  Restart your kernel/container completely")
print("   2️⃣  Reinstall PyTorch: pip install --force-reinstall torch")
print("   3️⃣  Check if your container needs CUDA runtime mounted")
print("   4️⃣  Use CPU mode temporarily: device = torch.device('cpu')")

7. Diagnosing PyTorch CUDA runtime libraries...
   Checking CUDA runtime libraries...
   ❌ No CUDA runtime libraries found

8. Testing PyTorch CUDA backend loading...
   ❌ PyTorch CUDA backend error: No module named 'torch._C._cuda'; 'torch._C' is not a package
   ❌ This suggests PyTorch CUDA bindings are broken

9. Potential solutions based on diagnosis:
   💡 Since NVIDIA drivers work but PyTorch CUDA fails:
   💡 This is likely a PyTorch CUDA runtime library issue
   💡 Solutions to try:
   1️⃣  Restart your kernel/container completely
   2️⃣  Reinstall PyTorch: pip install --force-reinstall torch
   3️⃣  Check if your container needs CUDA runtime mounted
   4️⃣  Use CPU mode temporarily: device = torch.device('cpu')


# 🔍 **DIAGNOSIS COMPLETE**

## **Root Cause Found:**

✅ **NVIDIA Drivers**: Working perfectly (all 4 GPUs detected)  
✅ **GPU Device Files**: All accessible with correct permissions  
✅ **NVML Access**: Direct GPU communication works  
❌ **PyTorch CUDA Runtime**: Missing/corrupted CUDA runtime libraries  
❌ **PyTorch CUDA Bindings**: Broken (`torch._C._cuda` not found)  

## **What Happened:**
Since you were working with these exact versions hours ago, something corrupted PyTorch's CUDA runtime libraries or bindings in your environment. This commonly happens due to:

1. **Environment corruption** during long-running experiments
2. **Partial package updates** that broke dependencies  
3. **Memory issues** during model loading that corrupted libraries
4. **Container state corruption** after intensive GPU usage

## **Immediate Fix:**
Since your experiments are done and you need CUDA working again, the fastest solution is:

### **Option 1: Restart Everything (Quickest)**
```bash
# Restart your container/kernel completely
# This often fixes corrupted runtime state
```

### **Option 2: Force Reinstall PyTorch**  
```bash
pip install --force-reinstall torch torchvision torchaudio
```

### **Option 3: Fresh Environment**
```bash
# Create a new conda environment with fresh PyTorch installation
conda create -n fresh_env python=3.11
conda activate fresh_env
pip install torch torchvision torchaudio
```

In [3]:
# Since PyTorch reinstall didn't work, let's check system-level issues
print("=== DEEPER SYSTEM DIAGNOSTICS ===\n")

# 1. Check if this is a container runtime issue
print("1. Container Runtime Analysis...")
try:
    # Check if nvidia-container-runtime is working
    result = subprocess.run(['docker', '--version'], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"   Docker version: {result.stdout.strip()}")
    
    # Check container capabilities
    with open('/proc/self/status', 'r') as f:
        for line in f:
            if 'CapEff' in line:
                print(f"   Container capabilities: {line.strip()}")
                break
                
except Exception as e:
    print(f"   Container check error: {e}")

print()

# 2. Test if CUDA works in a fresh Python process
print("2. Testing fresh Python subprocess...")
try:
    test_script = '''
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    x = torch.tensor([1.0], device="cuda:0")
    print("Tensor creation successful")
else:
    print("CUDA not available")
'''
    
    result = subprocess.run([
        '/opt/conda/envs/my_env/bin/python', '-c', test_script
    ], capture_output=True, text=True, timeout=30)
    
    print(f"   Fresh process stdout: {result.stdout}")
    if result.stderr:
        print(f"   Fresh process stderr: {result.stderr}")
    print(f"   Return code: {result.returncode}")
    
except Exception as e:
    print(f"   Fresh process test failed: {e}")

print()

# 3. Check CUDA context and driver state more deeply
print("3. Advanced CUDA driver state...")
try:
    # Check nvidia-smi for any error states
    result = subprocess.run(['nvidia-smi', '-q', '-d', 'MEMORY,COMPUTE'], 
                          capture_output=True, text=True, check=True)
    
    # Look for error patterns in output
    output_lines = result.stdout.split('\n')
    for line in output_lines:
        if any(keyword in line.lower() for keyword in ['error', 'failed', 'unavailable']):
            print(f"   ⚠️  {line.strip()}")
    
    print("   ✅ nvidia-smi detailed query successful")
            
except Exception as e:
    print(f"   ❌ nvidia-smi detailed query failed: {e}")

print()

# 4. Check if we can access CUDA through ctypes directly
print("4. Direct CUDA library access test...")
try:
    import ctypes
    # Try to load CUDA runtime directly
    try:
        cuda_rt = ctypes.CDLL('libcudart.so')
        print("   ✅ libcudart.so loaded directly")
        
        # Try to call cudaGetDeviceCount
        device_count = ctypes.c_int()
        result = cuda_rt.cudaGetDeviceCount(ctypes.byref(device_count))
        print(f"   ✅ Direct cudaGetDeviceCount: {device_count.value} devices, error code: {result}")
        
    except Exception as lib_e:
        print(f"   ❌ Direct CUDA library access failed: {lib_e}")
        
except Exception as e:
    print(f"   ❌ ctypes test failed: {e}")

print()

# 5. Check if this is a LD_LIBRARY_PATH issue
print("5. Library path analysis...")
ld_path = os.environ.get('LD_LIBRARY_PATH', 'Not set')
print(f"   LD_LIBRARY_PATH: {ld_path}")

# Check where PyTorch is looking for libraries
import torch
torch_path = torch.__file__
print(f"   PyTorch location: {torch_path}")

# Check the actual torch installation
try:
    import torch.version
    print(f"   PyTorch build info: {torch.version}")
except:
    pass

=== DEEPER SYSTEM DIAGNOSTICS ===

1. Container Runtime Analysis...
   Container check error: [Errno 2] No such file or directory: 'docker'

2. Testing fresh Python subprocess...
   Fresh process stdout: CUDA available: False
CUDA not available

   Fresh process stderr: /opt/conda/envs/my_env/lib/python3.11/site-packages/torch/cuda/__init__.py:182: UserWarning: CUDA initialization: CUDA driver initialization failed, you might not have a CUDA gpu. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0

   Return code: 0

3. Advanced CUDA driver state...
   ✅ nvidia-smi detailed query successful

4. Direct CUDA library access test...
   ❌ Direct CUDA library access failed: libcudart.so: cannot open shared object file: No such file or directory

5. Library path analysis...
   LD_LIBRARY_PATH: /usr/local/nvidia/lib:/usr/local/nvidia/lib64
   PyTorch location: /opt/conda/envs/my_env/lib/python3.11/site-packages/torch/__init__.py
   Py

In [2]:
# GHOST PYTORCH INVESTIGATION
# You uninstalled from both environments but import still works
print("🕵️ INVESTIGATING GHOST PYTORCH INSTALLATION...\n")

import torch
import sys

print("1. PyTorch Import Analysis:")
print(f"   torch module: {torch}")
print(f"   torch.__file__: {getattr(torch, '__file__', '❌ MISSING')}")

# Check critical PyTorch attributes
critical_attrs = ['__version__', 'cuda', 'tensor', 'device', 'nn']
print(f"\n2. Critical PyTorch Attributes:")
missing = []
for attr in critical_attrs:
    if hasattr(torch, attr):
        print(f"   ✅ torch.{attr}")
    else:
        print(f"   ❌ torch.{attr} MISSING")
        missing.append(attr)

print(f"\n3. Diagnosis:")
if missing:
    print(f"   🚨 CORRUPTED INSTALLATION: Missing {len(missing)} critical attributes")
    print(f"   This is a 'ghost' PyTorch - imports but doesn't work")
    print(f"   Missing: {missing}")
else:
    print(f"   ✅ PyTorch seems intact, checking CUDA...")
    try:
        print(f"   torch.__version__: {torch.__version__}")
        print(f"   torch.cuda.is_available(): {torch.cuda.is_available()}")
    except Exception as e:
        print(f"   ❌ Error accessing PyTorch functions: {e}")

print(f"\n4. 💡 SOLUTION for Ghost PyTorch:")
if missing:
    print("   Since uninstalling didn't work, you have corrupted remnants")
    print("   🔧 MANUAL CLEANUP NEEDED:")
    print("   1. Find torch directories with: find /opt/conda -name '*torch*' 2>/dev/null")
    print("   2. Delete them manually: rm -rf <torch_directories>")
    print("   3. Clear pip cache: rm -rf ~/.cache/pip")
    print("   4. Restart kernel completely")
    print("   5. Fresh install: pip install torch torchvision torchaudio")

🕵️ INVESTIGATING GHOST PYTORCH INSTALLATION...

1. PyTorch Import Analysis:
   torch module: <module 'torch' (<_frozen_importlib_external.NamespaceLoader object at 0x7f1d1c2b7490>)>
   torch.__file__: None

2. Critical PyTorch Attributes:
   ❌ torch.__version__ MISSING
   ❌ torch.cuda MISSING
   ❌ torch.tensor MISSING
   ❌ torch.device MISSING
   ❌ torch.nn MISSING

3. Diagnosis:
   🚨 CORRUPTED INSTALLATION: Missing 5 critical attributes
   This is a 'ghost' PyTorch - imports but doesn't work
   Missing: ['__version__', 'cuda', 'tensor', 'device', 'nn']

4. 💡 SOLUTION for Ghost PyTorch:
   Since uninstalling didn't work, you have corrupted remnants
   🔧 MANUAL CLEANUP NEEDED:
   1. Find torch directories with: find /opt/conda -name '*torch*' 2>/dev/null
   2. Delete them manually: rm -rf <torch_directories>
   3. Clear pip cache: rm -rf ~/.cache/pip
   4. Restart kernel completely
   5. Fresh install: pip install torch torchvision torchaudio


In [3]:
# MANUAL CLEANUP OF GHOST PYTORCH
print("🧹 FINDING GHOST PYTORCH FILES TO DELETE...\n")

import subprocess
import os

# 1. Find all torch-related directories and files
print("1. Searching for torch remnants...")
search_commands = [
    "find /opt/conda -name '*torch*' 2>/dev/null | head -20",
    "find /usr/local -name '*torch*' 2>/dev/null | head -10",
    "find ~/.local -name '*torch*' 2>/dev/null | head -10",
]

all_torch_files = []
for cmd in search_commands:
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if result.stdout.strip():
            files = result.stdout.strip().split('\n')
            all_torch_files.extend(files)
            print(f"   Found {len(files)} items in {cmd.split()[1]}:")
            for f in files[:5]:  # Show first 5
                print(f"     {f}")
            if len(files) > 5:
                print(f"     ... and {len(files)-5} more")
    except Exception as e:
        print(f"   Error with {cmd}: {e}")

print(f"\n2. Summary: Found {len(all_torch_files)} torch-related items")

# 2. Check Python import paths for torch
print(f"\n3. Checking Python sys.path for torch...")
import sys
for i, path in enumerate(sys.path):
    if os.path.exists(path):
        try:
            items = os.listdir(path)
            torch_items = [item for item in items if 'torch' in item.lower()]
            if torch_items:
                print(f"   Path {i}: {path}")
                print(f"     Torch items: {torch_items}")
        except:
            pass

print(f"\n4. 🔧 CLEANUP COMMANDS:")
print("   Run these commands to clean up ghost PyTorch:")
print("   ")
print("   # Stop this notebook/kernel first, then run in terminal:")
print("   # Remove all torch directories found")
if all_torch_files:
    unique_dirs = set()
    for f in all_torch_files:
        if '/site-packages/' in f:
            # Get the torch directory in site-packages
            parts = f.split('/site-packages/')
            if len(parts) > 1:
                base = parts[0] + '/site-packages'
                torch_dir = parts[1].split('/')[0]
                if 'torch' in torch_dir.lower():
                    full_path = f"{base}/{torch_dir}"
                    unique_dirs.add(full_path)
    
    for dir_path in sorted(unique_dirs):
        print(f"   rm -rf '{dir_path}'")

print(f"   ")
print("   # Clear caches")
print("   rm -rf ~/.cache/pip")
print("   rm -rf ~/.cache/torch")
print("   ")
print("   # Fresh install")  
print("   pip install torch torchvision torchaudio")
print("   ")
print("   # Then restart this notebook")

print(f"\n5. ⚠️  IMPORTANT:")
print("   - Stop this kernel/notebook before running cleanup commands")
print("   - The ghost PyTorch is preventing proper installation")
print("   - Manual cleanup is necessary because pip/conda can't see the broken installation")

🧹 FINDING GHOST PYTORCH FILES TO DELETE...

1. Searching for torch remnants...
   Found 20 items in /opt/conda:
     /opt/conda/envs/pdf-extract-kit-1.0/bin/torchrun
     /opt/conda/envs/pdf-extract-kit-1.0/lib/python3.10/site-packages/sympy/printing/__pycache__/pytorch.cpython-310.pyc
     /opt/conda/envs/pdf-extract-kit-1.0/lib/python3.10/site-packages/sympy/printing/pytorch.py
     /opt/conda/envs/pdf-extract-kit-1.0/lib/python3.10/site-packages/sympy/printing/tests/__pycache__/test_torch.cpython-310.pyc
     /opt/conda/envs/pdf-extract-kit-1.0/lib/python3.10/site-packages/sympy/printing/tests/test_torch.py
     ... and 15 more

2. Summary: Found 20 torch-related items

3. Checking Python sys.path for torch...
   Path 5: /opt/conda/envs/my_env/lib/python3.11/site-packages
     Torch items: ['torch', 'torchtext-0.18.0.dist-info', 'torchtext']

4. 🔧 CLEANUP COMMANDS:
   Run these commands to clean up ghost PyTorch:
   
   # Stop this notebook/kernel first, then run in terminal:
   # R

# 🎯 **SOLUTION: Clean Up Ghost PyTorch**

## **Problem Identified:**
- You have a **ghost PyTorch installation** in `/opt/conda/envs/my_env/lib/python3.11/site-packages/torch`
- It imports successfully but has **no attributes** (`__version__`, `cuda`, etc.)
- This happens when uninstallation was incomplete

## **Root Cause:**
Your PyTorch experiments created a corrupted installation that:
1. ✅ Can be imported (`import torch` works)
2. ❌ Has no functionality (missing all attributes)
3. ❌ Invisible to pip/conda (so reinstall doesn't work)

## **Immediate Fix:**

### **Step 1: Stop this notebook/kernel**
### **Step 2: Run these commands in terminal:**

```bash
# Remove the ghost PyTorch directory
rm -rf /opt/conda/envs/my_env/lib/python3.11/site-packages/torch
rm -rf /opt/conda/envs/my_env/lib/python3.11/site-packages/torchtext*

# Clear all caches
rm -rf ~/.cache/pip
rm -rf ~/.cache/torch

# Activate your environment and fresh install
conda activate my_env
pip install torch torchvision torchaudio
```

### **Step 3: Restart this notebook and test**

This will completely remove the corrupted PyTorch and give you a fresh, working installation that should resolve your CUDA initialization errors.

In [4]:
# DOCKER/CONTAINER CUDA DIAGNOSTICS
# Since PyTorch reinstall worked but CUDA still fails, this is a container issue
print("🐳 DOCKER CONTAINER CUDA DIAGNOSTICS...\n")

import subprocess
import os

print("1. Container Runtime Information:")
try:
    # Check if we're in Docker
    with open('/.dockerenv', 'r') as f:
        print("   ✅ Running inside Docker container")
except FileNotFoundError:
    print("   ❓ Not a standard Docker container (/.dockerenv missing)")

# Check container hostname
try:
    hostname = subprocess.run(['hostname'], capture_output=True, text=True).stdout.strip()
    print(f"   Container hostname: {hostname}")
except:
    pass

print("\n2. NVIDIA Container Runtime Check:")
try:
    # Check if nvidia-container-runtime-hook exists
    nvidia_hook_paths = [
        '/usr/bin/nvidia-container-runtime-hook',
        '/usr/local/bin/nvidia-container-runtime-hook'
    ]
    
    for path in nvidia_hook_paths:
        if os.path.exists(path):
            print(f"   ✅ Found nvidia-container-runtime-hook at {path}")
        else:
            print(f"   ❌ Missing {path}")
            
    # Check nvidia-container-cli
    try:
        result = subprocess.run(['nvidia-container-cli', '--version'], 
                              capture_output=True, text=True, timeout=5)
        if result.returncode == 0:
            print(f"   ✅ nvidia-container-cli version: {result.stdout.strip()}")
        else:
            print(f"   ❌ nvidia-container-cli failed: {result.stderr}")
    except FileNotFoundError:
        print("   ❌ nvidia-container-cli not found")
    except Exception as e:
        print(f"   ❌ nvidia-container-cli error: {e}")
        
except Exception as e:
    print(f"   Error checking NVIDIA container runtime: {e}")

print("\n3. Container GPU Mount Points:")
# Check if GPU devices are properly mounted
gpu_mounts = [
    '/dev/nvidia0', '/dev/nvidia1', '/dev/nvidia2', '/dev/nvidia3',
    '/dev/nvidiactl', '/dev/nvidia-uvm', '/dev/nvidia-uvm-tools'
]

mounted_correctly = 0
for device in gpu_mounts:
    if os.path.exists(device):
        try:
            stat = os.stat(device)
            # Check if it's a character device (should be for GPU devices)
            import stat as stat_module
            if stat_module.S_ISCHR(stat.st_mode):
                print(f"   ✅ {device} (character device)")
                mounted_correctly += 1
            else:
                print(f"   ⚠️  {device} exists but not a character device")
        except Exception as e:
            print(f"   ❌ {device}: {e}")
    else:
        print(f"   ❌ {device} missing")

print(f"\n   Summary: {mounted_correctly}/{len(gpu_mounts)} GPU devices properly mounted")

print("\n4. Docker Environment Variables:")
# Check Docker/container specific environment variables
docker_env_vars = [
    'NVIDIA_VISIBLE_DEVICES',
    'NVIDIA_DRIVER_CAPABILITIES', 
    'NVIDIA_REQUIRE_CUDA',
    'CUDA_VISIBLE_DEVICES'
]

for var in docker_env_vars:
    value = os.environ.get(var, 'Not set')
    status = "✅" if value != 'Not set' else "❌"
    print(f"   {status} {var}: {value}")

print("\n5. Container Capabilities:")
try:
    with open('/proc/self/status', 'r') as f:
        for line in f:
            if 'Cap' in line and ('Eff' in line or 'Permitted' in line):
                print(f"   {line.strip()}")
except Exception as e:
    print(f"   Error reading capabilities: {e}")

print("\n6. 🔧 LIKELY DOCKER ISSUES:")
print("   Common container CUDA problems:")
print("   1️⃣  Container started without --gpus flag")
print("   2️⃣  NVIDIA Docker runtime not available")
print("   3️⃣  Container lacks proper device mounts")
print("   4️⃣  Missing NVIDIA_VISIBLE_DEVICES environment variable")
print("   5️⃣  Container capabilities don't allow GPU access")

🐳 DOCKER CONTAINER CUDA DIAGNOSTICS...

1. Container Runtime Information:
   ✅ Running inside Docker container
   Container hostname: rftInsideDocker

2. NVIDIA Container Runtime Check:
   ❌ Missing /usr/bin/nvidia-container-runtime-hook
   ❌ Missing /usr/local/bin/nvidia-container-runtime-hook
   ❌ nvidia-container-cli not found

3. Container GPU Mount Points:
   ✅ /dev/nvidia0 (character device)
   ✅ /dev/nvidia1 (character device)
   ✅ /dev/nvidia2 (character device)
   ✅ /dev/nvidia3 (character device)
   ✅ /dev/nvidiactl (character device)
   ✅ /dev/nvidia-uvm (character device)
   ✅ /dev/nvidia-uvm-tools (character device)

   Summary: 7/7 GPU devices properly mounted

4. Docker Environment Variables:
   ✅ NVIDIA_VISIBLE_DEVICES: all
   ✅ NVIDIA_DRIVER_CAPABILITIES: compute,utility
   ❌ NVIDIA_REQUIRE_CUDA: Not set
   ❌ CUDA_VISIBLE_DEVICES: Not set

5. Container Capabilities:
   CapEff:	00000000a80425fb

6. 🔧 LIKELY DOCKER ISSUES:
   Common container CUDA problems:
   1️⃣  Conta

In [5]:
# CONTAINER CUDA RUNTIME LIBRARIES CHECK
print("🔍 CONTAINER CUDA RUNTIME ANALYSIS...\n")

print("7. CUDA Library Path Issues:")
# Check LD_LIBRARY_PATH in container
ld_path = os.environ.get('LD_LIBRARY_PATH', '')
print(f"   LD_LIBRARY_PATH: {ld_path}")

# Check if CUDA runtime libraries are accessible
cuda_lib_locations = [
    '/usr/local/nvidia/lib64',
    '/usr/local/nvidia/lib', 
    '/usr/local/cuda/lib64',
    '/usr/lib/x86_64-linux-gnu'
]

print("   Checking CUDA library locations:")
found_cudart = False
for location in cuda_lib_locations:
    if os.path.exists(location):
        try:
            files = os.listdir(location)
            cudart_files = [f for f in files if 'cudart' in f]
            if cudart_files:
                print(f"   ✅ {location}: {cudart_files}")
                found_cudart = True
            else:
                print(f"   ⚠️  {location}: exists but no cudart")
        except Exception as e:
            print(f"   ❌ {location}: {e}")
    else:
        print(f"   ❌ {location}: does not exist")

if not found_cudart:
    print("   🚨 NO CUDART LIBRARIES FOUND IN CONTAINER!")

print("\n8. Direct Library Loading Test:")
try:
    import ctypes
    
    # Try different ways to load CUDA runtime
    lib_attempts = [
        'libcudart.so.12',
        'libcudart.so.11',  
        'libcudart.so',
        '/usr/local/nvidia/lib64/libcudart.so.12'
    ]
    
    loaded = False
    for lib_name in lib_attempts:
        try:
            lib = ctypes.CDLL(lib_name)
            print(f"   ✅ Successfully loaded {lib_name}")
            
            # Try to get device count
            device_count = ctypes.c_int()
            result = lib.cudaGetDeviceCount(ctypes.byref(device_count))
            print(f"   ✅ cudaGetDeviceCount: {device_count.value} devices (error: {result})")
            loaded = True
            break
        except Exception as e:
            print(f"   ❌ Failed to load {lib_name}: {e}")
    
    if not loaded:
        print("   🚨 COULD NOT LOAD ANY CUDA RUNTIME LIBRARY!")
        
except Exception as e:
    print(f"   Error in library loading test: {e}")

print("\n9. Container Restart Recommendation:")
if not found_cudart:
    print("   🎯 ROOT CAUSE IDENTIFIED:")
    print("   Your container is missing CUDA runtime libraries!")
    print("   ")
    print("   💡 SOLUTIONS:")
    print("   1️⃣  Restart container with proper nvidia runtime:")
    print("      docker run --gpus all --runtime=nvidia ...")
    print("   ")
    print("   2️⃣  Use nvidia/cuda base image:")
    print("      FROM nvidia/cuda:12.1-runtime-ubuntu20.04")
    print("   ")
    print("   3️⃣  Install CUDA runtime in current container:")
    print("      apt update && apt install -y cuda-runtime-12-1")
    print("   ")
    print("   4️⃣  Mount CUDA libraries from host:")
    print("      -v /usr/local/cuda/lib64:/usr/local/cuda/lib64:ro")
else:
    print("   ✅ CUDA libraries found - issue may be elsewhere")
    print("   Try restarting container or checking PyTorch CUDA compilation")

🔍 CONTAINER CUDA RUNTIME ANALYSIS...

7. CUDA Library Path Issues:
   LD_LIBRARY_PATH: /usr/local/nvidia/lib:/usr/local/nvidia/lib64
   Checking CUDA library locations:
   ❌ /usr/local/nvidia/lib64: does not exist
   ❌ /usr/local/nvidia/lib: does not exist
   ❌ /usr/local/cuda/lib64: does not exist
   ⚠️  /usr/lib/x86_64-linux-gnu: exists but no cudart
   🚨 NO CUDART LIBRARIES FOUND IN CONTAINER!

8. Direct Library Loading Test:
   ✅ Successfully loaded libcudart.so.12
   ✅ cudaGetDeviceCount: 0 devices (error: 3)

9. Container Restart Recommendation:
   🎯 ROOT CAUSE IDENTIFIED:
   Your container is missing CUDA runtime libraries!
   
   💡 SOLUTIONS:
   1️⃣  Restart container with proper nvidia runtime:
      docker run --gpus all --runtime=nvidia ...
   
   2️⃣  Use nvidia/cuda base image:
      FROM nvidia/cuda:12.1-runtime-ubuntu20.04
   
   3️⃣  Install CUDA runtime in current container:
      apt update && apt install -y cuda-runtime-12-1
   
   4️⃣  Mount CUDA libraries from host